In [3]:
from transformers import pipeline
from transformers import RobertaModel, RobertaTokenizer, RobertaForSequenceClassification
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


c:\Users\danie\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "gpt2"
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = RobertaTokenizer.from_pretrained(model_name)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'RobertaTokenizer'.


In [3]:
prompt = "Given the following events, extract the potential causes for each:\n\nEvent: The server crashed. Cause:\nEvent: The meeting was canceled. Cause:\nEvent: The project was delayed. Cause:\n\nExample: Event: The traffic accident occurred. Cause: Poor weather conditions and reckless driving.\n\nNotes:\n"

input_texts = [
    "The server crashed due to a power outage.",
    "The meeting was canceled because of scheduling conflicts.",
    "The project was delayed caused by unexpected issues and resource constraints."
]

In [4]:
input_texts = ["The server crashed due to a power outage."]

In [5]:
inputs = tokenizer(prompt + input_texts[0], return_tensors="pt", truncation=True, max_length=512)


In [6]:
outputs = model(**inputs)
potential_causes = tokenizer.decode(outputs.logits.argmax(-1))

IndexError: index out of range in self

In [35]:
print("Potential causes:")
print(potential_causes)

Potential causes:
<s>


In [4]:
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli")

sequence_to_classify = "On 17 July 2023, workers picketed in Lawrence Park (Pennsylvania) to call for a better contract from Wabtec that includes a cost-of-living adjustment and wage increases. The picket took place as part of a strike which began on 22 June 2023 and was organized by UE Local 506 and 618. During the picket, two picketers were hit by a car in a hit-and-run, though there were no serious injuries reported."

candidate_labels = ["social-justice", "wages", "violence", "healthcare"]
output = classifier(sequence_to_classify, candidate_labels, multi_label=False)
print(output)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'sequence': 'On 17 July 2023, workers picketed in Lawrence Park (Pennsylvania) to call for a better contract from Wabtec that includes a cost-of-living adjustment and wage increases. The picket took place as part of a strike which began on 22 June 2023 and was organized by UE Local 506 and 618. During the picket, two picketers were hit by a car in a hit-and-run, though there were no serious injuries reported.', 'labels': ['violence', 'wages', 'social-justice', 'healthcare'], 'scores': [0.6506263017654419, 0.3436485528945923, 0.0042635295540094376, 0.0014616013504564762]}


In [10]:
def get_labels(notes, model):
    labels = []

    classifier = pipeline("zero-shot-classification", model=model)

    for note in notes:
        sequence_to_classify = note[0]  # Extracting the note string from the inner list
        output = classifier(sequence_to_classify, candidate_labels, multi_label=False)
        # Store scores for this note separately
        scores_for_note = output["scores"]
        labels.append(scores_for_note)

    return labels



In [11]:
event_notes = [["On 17 July 2023, workers picketed in Lawrence Park (Pennsylvania) to call for a better contract from Wabtec that includes a cost-of-living adjustment and wage increases. The picket took place as part of a strike which began on 22 June 2023 and was organized by UE Local 506 and 618. During the picket, two picketers were hit by a car in a hit-and-run, though there were no serious injuries reported."],
            ["On 2 June 2023, more than two dozen people gathered for a National Gun Violence Awareness Day rally at the Montgomery County Courthouse in Norristown (Pennsylvania) to demand 'common sense' gun legislation. CeaseFirePA was present at the event. State Sen. Amanda Cappelletti (D), State Rep. Greg Scott (D), and U.S. Congresswoman Madeleine Dean (D) spoke at the event."]]


model = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

labels = get_labels(event_notes, model)



Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [12]:
print(labels)

[[0.6506263017654419, 0.3436485528945923, 0.0042635295540094376, 0.0014616013504564762], [0.9052410125732422, 0.057785797864198685, 0.032166529446840286, 0.004806684795767069]]


In [13]:
import pandas as pd
acled_notes = pd.read_excel("notes.xlsx")

In [14]:
acled_notes = acled_notes["notes"].tolist()

In [15]:
acled_labels = get_labels(acled_notes, model)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [18]:
print(acled_labels)

acled_labels = pd.DataFrame(acled_labels)

[[0.28717249631881714, 0.2523346245288849, 0.24925927817821503, 0.21123361587524414], [0.28717249631881714, 0.2523346245288849, 0.24925927817821503, 0.21123361587524414], [0.28717249631881714, 0.2523346245288849, 0.24925927817821503, 0.21123361587524414], [0.28717249631881714, 0.2523346245288849, 0.24925927817821503, 0.21123361587524414], [0.28717249631881714, 0.2523346245288849, 0.24925927817821503, 0.21123361587524414], [0.28717249631881714, 0.2523346245288849, 0.24925927817821503, 0.21123361587524414], [0.28717249631881714, 0.2523346245288849, 0.24925927817821503, 0.21123361587524414], [0.28717249631881714, 0.2523346245288849, 0.24925927817821503, 0.21123361587524414], [0.28717249631881714, 0.2523346245288849, 0.24925927817821503, 0.21123361587524414], [0.28717249631881714, 0.2523346245288849, 0.24925927817821503, 0.21123361587524414], [0.28717249631881714, 0.2523346245288849, 0.24925927817821503, 0.21123361587524414], [0.28717249631881714, 0.2523346245288849, 0.24925927817821503, 0